# Báo cáo Project
Lớp 172886, Nhóm N07

## 1. Thông tin chung

### Thành viên

| STT | Họ và tên | MSSV |
|-----|-----------|------|
| 1 | Nguyễn Ngọc Quang | 202516236 |
| 2 | Nguyễn Việt Khánh | 202516138 |
| 3 | Nguyễn Quang Trung | 202516277 |
| 4 | Cao Ngọc Trường Sơn | 202516243 |

### Lịch thực hiện
- W2: Đăng ký nhóm 
- W3-4: Đề xuất project (26/9)
- W08: Báo cáo tiến độ giữa kỳ (24/10)
- W15: Hoàn thành và gửi báo cáo project (12/12)
- W16-18: Trình bày project, Q&A

## 2. Đề xuất project (W3)

### Bài toán

**Sokoban** là trò chơi giải đố trên lưới: người chơi phải đẩy tất cả các hộp về ô đích, chỉ được **đẩy** chứ không được kéo.

| Ký tự | Ý nghĩa |
|-------|---------|
| `#` | Tường |
| `.` | Ô đích |
| `$` | Hộp |
| `*` | Hộp trên đích |
| `@` | Người chơi |
| `+` | Người chơi trên đích |

- **Trạng thái**: vị trí người chơi + tập vị trí các hộp.
- **Hành động**: đi 1 ô theo 4 hướng; nếu ô trước là hộp và ô sau hộp trống thì đẩy hộp. Chi phí mỗi bước bằng 1.
- **Trạng thái đích**: mọi hộp nằm trên ô đích.
- **Mục tiêu**: tìm lời giải ít bước nhất, đồng thời đo chi phí tìm kiếm (số node mở rộng, thời gian, bộ nhớ).

Sokoban là bài toán **PSPACE-complete**, không gian trạng thái rất lớn. Vì vậy project so sánh **BFS** (tìm kiếm không heuristic) với **A\*** (có heuristic và cắt nhánh deadlock) trên cùng một bộ level.

### Phương pháp dự kiến

- **BFS (Breadth-First Search)**: duyệt theo chiều rộng, không dùng tri thức. Đảm bảo tìm lời giải tối ưu nhưng bộ nhớ tăng rất nhanh.
- **A\***: dùng hàm $f(n) = g(n) + h(n)$, với $g(n)$ là số bước đã đi và $h(n)$ là heuristic ước lượng chi phí còn lại.
  - Heuristic: khoảng cách Manhattan từ hộp tới ô đích gần nhất (và từ người chơi tới hộp cần đẩy).
  - Kỹ thuật cắt nhánh: phát hiện **deadlock** (hộp bị kẹt ở góc hoặc không thể tới đích) để loại bỏ nhánh vô nghiệm.
- **Cài đặt chung**: biểu diễn trạng thái bằng vị trí người chơi + frozenset vị trí hộp, sinh trạng thái kế từ 4 hướng di chuyển, chạy trên cùng một bộ level để so sánh công bằng.
- **Đánh giá**: số node mở rộng, thời gian chạy, bộ nhớ, độ dài lời giải và tỉ lệ level giải được.

### Phân công
-
-
-
-

## 3. Tiến độ giữa kỳ (W8)

### Chương trình

Đến mốc giữa kỳ, nhóm đã xây dựng được khung chương trình:

- Đọc bàn cờ từ file văn bản, xác định tường, ô đích, hộp và vị trí người chơi.
- Biểu diễn trạng thái (vị trí người chơi + tập vị trí hộp) và sinh trạng thái kế từ 4 hướng di chuyển, kiểm tra nước đi hợp lệ.
- Cài đặt **BFS**: kiểm tra trạng thái đích, truy vết và in ra chuỗi lời giải.
- Chương trình chạy được trên các level nhỏ (ít hộp, bàn cờ nhỏ).

In [1]:
# Biểu diễn bàn cờ và sinh trạng thái kế

# Ký hiệu: # tường | . đích | $ hộp | * hộp trên đích | @ người chơi | + người chơi trên đích
DIRS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}


class Level:
    """Bàn cờ Sokoban đọc từ văn bản."""

    def __init__(self, name, text):
        self.name = name
        lines = text.strip("\n").split("\n")
        self.height = len(lines)
        self.width = max(len(line) for line in lines)
        self.walls, self.goals = set(), set()
        boxes, self.player = set(), None
        for r, line in enumerate(lines):
            for c, ch in enumerate(line):
                if ch == "#":
                    self.walls.add((r, c))
                if ch in ".*+":
                    self.goals.add((r, c))
                if ch in "$*":
                    boxes.add((r, c))
                if ch in "@+":
                    self.player = (r, c)
        self.boxes = frozenset(boxes)
        assert self.player is not None, "Level thiếu người chơi (@)"

    def is_goal(self, boxes):
        """Mọi hộp đều nằm trên ô đích."""
        return set(boxes) <= self.goals

    def successors(self, player, boxes):
        """Sinh cặp (hành động, trạng thái kế) cho 4 hướng."""
        for mv, (dr, dc) in DIRS.items():
            r, c = player[0] + dr, player[1] + dc
            if (r, c) in self.walls:
                continue
            if (r, c) in boxes:                   # phía trước là hộp -> đẩy
                br, bc = r + dr, c + dc
                if (br, bc) in self.walls or (br, bc) in boxes:
                    continue                      # ô sau hộp không trống
                yield mv, ((r, c), frozenset(boxes - {(r, c)} | {(br, bc)}))
            else:                                 # ô trống -> đi bộ
                yield mv, ((r, c), boxes)

    def render(self, player, boxes):
        """Vẽ bàn cờ tại một trạng thái."""
        rows = []
        for r in range(self.height):
            row = []
            for c in range(self.width):
                p = (r, c)
                if p in self.walls:
                    ch = "#"
                elif p in boxes:
                    ch = "*" if p in self.goals else "$"
                elif p == player:
                    ch = "+" if p in self.goals else "@"
                elif p in self.goals:
                    ch = "."
                else:
                    ch = " "
                row.append(ch)
            rows.append("".join(row))
        return "\n".join(rows)

In [2]:
# BFS: duyệt theo chiều rộng
from collections import deque
import time


def reconstruct(parent, action, state):
    """Truy vết lời giải từ trạng thái đích về trạng thái đầu."""
    moves = []
    while parent[state] is not None:
        moves.append(action[state])
        state = parent[state]
    return "".join(reversed(moves))


def bfs_solve(level, max_nodes=300_000, time_limit=60.0):
    """Trả về (lời giải, thống kê); lời giải = None nếu vượt giới hạn."""
    start = (level.player, level.boxes)
    stats = {"nodes": 0, "max_frontier": 0, "time": 0.0}
    t0 = time.perf_counter()
    if level.is_goal(level.boxes):
        stats["time"] = time.perf_counter() - t0
        return "", stats

    frontier = deque([start])
    parent, action = {start: None}, {}
    while frontier:
        if stats["nodes"] >= max_nodes or time.perf_counter() - t0 > time_limit:
            break
        player, boxes = frontier.popleft()
        stats["nodes"] += 1
        for mv, (p2, b2) in level.successors(player, boxes):
            ns = (p2, b2)
            if ns in parent:                  # trạng thái đã thăm
                continue
            parent[ns] = (player, boxes)
            action[ns] = mv
            if level.is_goal(b2):
                stats["time"] = time.perf_counter() - t0
                return reconstruct(parent, action, ns), stats
            frontier.append(ns)
        stats["max_frontier"] = max(stats["max_frontier"], len(frontier))
    stats["time"] = time.perf_counter() - t0
    return None, stats


# Ví dụ chạy trên một level nhỏ
demo = Level("demo", """
#######
#     #
# .$@ #
#     #
#######
""")
solution, st = bfs_solve(demo)
print("Trạng thái đầu:")
print(demo.render(demo.player, demo.boxes))
print(f"\nLời giải: {solution} | số bước: {len(solution)} | node đã mở rộng: {st['nodes']}")

Trạng thái đầu:
#######
#     #
# .$@ #
#     #
#######

Lời giải: L | số bước: 1 | node đã mở rộng: 1


### Bộ level và kết quả chạy thử

Bộ thử nghiệm gồm 11 level định dạng XSB: 2 level lấy từ bộ **Microban** (David W. Skinner), còn lại do nhóm thiết kế, độ khó tăng dần theo số hộp và độ mở của bàn cờ. Mỗi level chạy BFS với giới hạn **300.000 node** / **60 giây**; chương trình ghi lại số bước, node mở rộng, frontier lớn nhất, thời gian và bộ nhớ đỉnh (đo bằng `tracemalloc`).

In [3]:
LEVELS = {
    "L1-mot-hop": "#######\n#     #\n# @$. #\n#     #\n#######",
    "L2-day-doc": "#######\n#     #\n#  .  #\n#     #\n# @$  #\n#     #\n#######",
    "L3-hai-hop": "########\n#      #\n# .  . #\n#      #\n# $  $ #\n#  @   #\n########",
    "L4-goc": "#######\n#.    #\n#  #  #\n#  $  #\n#  @  #\n#     #\n#######",
    "L5-ba-hop": "#########\n#  . .  #\n#       #\n# $ @ $ #\n#   $   #\n#   .   #\n#########",
    "L6-hanh-lang": "#########\n#   #   #\n# . # . #\n#   #   #\n# $   $ #\n#   @   #\n#########",
    "L7-phong": "##########\n#   ..   #\n#        #\n#  $$    #\n#    @   #\n#        #\n##########",
    "L8-microban1": "####\n# .#\n#  ###\n#*@  #\n#  $ #\n#  ###\n####",
    "L9-microban2": "######\n#    #\n# #@ #\n# $* #\n# .* #\n#    #\n######",
    "L10-kho": "#########\n#  .#   #\n#  $$   #\n#   @   #\n#  .#   #\n#########",
    "L11-bon-hop": "##########\n#  .  .  #\n#        #\n#  $  $  #\n#   @    #\n#  $  $  #\n#  .  .  #\n##########",
}

In [4]:
import tracemalloc

MAX_NODES, TIME_LIMIT = 300_000, 60.0


def benchmark_bfs(level):
    """Chạy BFS hai lần: một lần đo thời gian, một lần đo bộ nhớ đỉnh."""
    t0 = time.perf_counter()
    solution, stats = bfs_solve(level, max_nodes=MAX_NODES, time_limit=TIME_LIMIT)
    elapsed = time.perf_counter() - t0

    tracemalloc.start()
    bfs_solve(level, max_nodes=MAX_NODES, time_limit=TIME_LIMIT)
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    return (level.name, len(level.boxes), solution is not None,
            len(solution) if solution is not None else None,
            stats["nodes"], stats["max_frontier"], elapsed, peak / 1e6)


rows = [benchmark_bfs(Level(name, text)) for name, text in LEVELS.items()]

print(f"{'Level':<14}{'Hộp':>4}{'Giải':>7}{'Bước':>6}{'Node':>9}"
      f"{'Frontier':>10}{'Giây':>8}{'MB':>7}")
print("-" * 65)
for name, boxes, solved, moves, nodes, frontier, seconds, mb in rows:
    print(f"{name:<14}{boxes:>4}{'có' if solved else 'không':>7}{moves:>6}"
          f"{nodes:>9,}{frontier:>10,}{seconds:>8.3f}{mb:>7.2f}")

print(f"\nTổng: {len(rows)} level, {sum(r[4] for r in rows):,} node | "
      f"thời gian lớn nhất {max(r[6] for r in rows):.3f}s | "
      f"bộ nhớ đỉnh lớn nhất {max(r[7] for r in rows):.1f} MB")

Level          Hộp   Giải  Bước     Node  Frontier    Giây     MB
-----------------------------------------------------------------
L1-mot-hop       1     có     1        1         0   0.000   0.00
L2-day-doc       1     có     4       20        19   0.000   0.00
L3-hai-hop       2     có    10      654       331   0.002   0.18
L4-goc           1     có     8      104        39   0.000   0.01
L5-ba-hop        3     có    20   89,083    24,187   0.442  36.77
L6-hanh-lang     2     có    12      807       292   0.001   0.20
L7-phong         2     có    12    3,022     1,745   0.008   1.28
L8-microban1     2     có    33      559        34   0.001   0.06
L9-microban2     3     có    16      901       361   0.002   0.27
L10-kho          2     có    14      603       189   0.001   0.13
L11-bon-hop      4     có    14   30,129    22,365   0.113  18.01

Tổng: 11 level, 125,883 node | thời gian lớn nhất 0.442s | bộ nhớ đỉnh lớn nhất 36.8 MB


### Phân tích kết quả, vấn đề gặp phải

- **Kết quả**: BFS giải được **11/11 level**, lời giải tối ưu về số bước (mọi bước có chi phí 1). Tổng cộng 125.883 node; nặng nhất là `L5-ba-hop` với 89.083 node và 36,8 MB (thời gian khoảng 0,3–0,5 giây tùy máy). Các level còn lại đều dưới 0,2 giây.
- **Chi phí phụ thuộc độ sâu lời giải và số trạng thái tới được, không phải kích thước bàn**: `L11-bon-hop` có 4 hộp nhưng chỉ tốn 30.129 node vì lời giải ngắn (14 bước); ngược lại `L8-microban1` có lời giải 33 bước nhưng chỉ 559 node vì bàn hẹp.
- **Vấn đề**: BFS phải giữ toàn bộ frontier trong bộ nhớ (lớn nhất 24.187 trạng thái ở `L5-ba-hop`) và không dùng tri thức nên duyệt đều mọi hướng — số node sẽ bùng nổ khi thêm hộp hoặc mở rộng bàn.
- **Chưa làm được**: A\*, heuristic và phát hiện deadlock.
- **Hướng khắc phục**: cài A\* với heuristic (khoảng cách Manhattan hoặc gán hộp–đích) và cắt nhánh ô chết; có thể tìm ở mức "lần đẩy" thay vì từng bước đi để thu nhỏ không gian trạng thái.

## 4. Cập nhật kết quả cuối kỳ (W15)

### Chi tiết phương pháp, dữ liệu 
....

### Chương trình
...

### Phân tích, đánh giá kết quả
...

### Cập nhật phân công, khối lượng công việc
<!-- công việc của các thành viên, tỷ lệ đóng góp của các thành viên -->
...